In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

train= pd.read_csv(r"C:\Users\parin\OneDrive\Desktop\dataset\triagist\triagegeist\train.csv")

test= pd.read_csv(r"C:\Users\parin\OneDrive\Desktop\dataset\triagist\triagegeist\test.csv")

print(train.shape)
train.head()

In [ ]:
cols_to_drop=['disposition', 'ed_los_hours']

In [ ]:
train_cleaned= train.drop(columns= cols_to_drop)

In [ ]:
train_cleaned.shape

In [ ]:
patient_history= pd.read_csv(r"C:\Users\parin\OneDrive\Desktop\dataset\triagist\triagegeist\patient_history.csv")
chief_complaints= pd.read_csv(r"C:\Users\parin\OneDrive\Desktop\dataset\triagist\triagegeist\chief_complaints.csv")

In [ ]:
from sklearn.preprocessing import LabelEncoder

le= LabelEncoder()

In [ ]:
chief_complaints.head()

In [ ]:
chief_complaints['chief_complaint_system_number'] = le.fit_transform(chief_complaints['chief_complaint_system'])



In [ ]:
train_merged = train.merge(patient_history, on='patient_id', how='left')
test_merged = test.merge(patient_history, on='patient_id', how='left')

In [ ]:
complaints_subset = chief_complaints[['patient_id', 'chief_complaint_raw']]
train_merged = train_merged.merge(complaints_subset, on='patient_id', how='left')
test_merged = test_merged.merge(complaints_subset, on='patient_id', how='left')

leakage_cols = ['ed_los_hours', 'disposition']
train_merged = train_merged.drop(columns=leakage_cols)



In [ ]:
!pip install torch transformers pandas numpy tqdm
import torch
from transformers import AutoTokenizer, AutoModel
import pandas as pd
import numpy as np
from tqdm import tqdm

model_name = "dmis-lab/biobert-v1.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
def get_bert_embeddings(text_list, batch_size=32):
    model.eval()  # set model to evaluation mode
    all_embeddings = []
    
    for i in tqdm(range(0, len(text_list), batch_size)):
        batch_texts = text_list[i:i+batch_size]
        
        # convert text → token IDs
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors="pt"
        ).to(device)
        
        with torch.no_grad():  # no gradients (faster, less memory)
            outputs = model(**inputs)
            
            # take CLS token (represents whole sentence)
            embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.append(embeddings)
    
    return np.vstack(all_embeddings)  # combine all batches

In [ ]:
texts = train_merged['chief_complaint_raw'].fillna("No Data").to_list()
bert_features = get_bert_embeddings(texts)

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=10) # 10次元程度に圧縮してモデルに入れやすくする
bert_pca = pca.fit_transform(bert_features)

In [ ]:
bert_pca

In [ ]:
bert_cols = [f'bert_pca_{i}' for i in range(10)]
print(bert_cols)
df_bert = pd.DataFrame(bert_pca, columns=bert_cols)
train_merged = pd.concat([train_merged, df_bert], axis=1)

In [ ]:
train = train_merged
target = 'triage_acuity'


In [ ]:
!pip install catboost scikit-learn torch transformers tqdm

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score
from sklearn.decomposition import PCA
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

In [ ]:
drop_cols = ['patient_id', 'triage_nurse_id', 'chief_complaint_raw', target]

In [ ]:
X = train.drop(columns=drop_cols)
y = train[target]

X.columns.values

In [ ]:
cat_features = X.select_dtypes(include=['object']).columns.tolist()
cat_features

In [ ]:
X[cat_features]


In [ ]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []
models = []

print("\nStarting Cross-Validation...")
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # CatBoost専用のデータセット形式
    train_pool = Pool(X_train, y_train, cat_features=cat_features)
    val_pool = Pool(X_val, y_val, cat_features=cat_features)
    
    model = CatBoostClassifier(
        iterations=10000,
        learning_rate=0.05,
        depth=6,
        loss_function='MultiClass',
        early_stopping_rounds=50,
        verbose=100,
        random_seed=42,
        task_type="GPU" if torch.cuda.is_available() else "CPU"
    )
    
    model.fit(train_pool, eval_set=val_pool)
    
    preds = model.predict(X_val)
    score = accuracy_score(y_val, preds)
    cv_scores.append(score)
    models.append(model)
    print(f"Fold {fold+1} Accuracy: {score:.4f}")

print(f"\nMean CV Accuracy: {np.mean(cv_scores):.4f}")

In [ ]:
import shap
import pandas as pd

# 1. SHAP値の計算 (学習済みの catboost モデルを使用)
# 注: model はクロスバリデーションで学習させたうちの1つ、または全データで再学習させたもの
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_val)
shap_values

In [ ]:
shap_values.shape


In [ ]:
X_val.shape


In [ ]:
shap_values[:,:,0].shape


In [ ]:
shap.summary_plot(shap_values[:,:,0], X_val) # 緊急度1に対する重要度
